# Hungarian Election Prediction 2026 | Andrási Kristóf - notebook 14


### Imports

This block loads the packages used in the notebook. The structure follows notebook 01 so the notebook opening stays consistent across the final folder.


In [1]:
# this block loads the packages used in the notebook.
import importlib
import subprocess
import sys

def ensure_packages(packages):
    # install a package only if it is missing
    for pkg in packages:
        try:
            importlib.import_module(pkg)
        except ImportError:
            print(f"{pkg} not found, installing...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])
            print(f"{pkg} installed.")

ensure_packages(["pandas", "numpy", "openpyxl", "matplotlib", "re", "xlrd", "scipy", "statsmodels", "IPython"])

import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import re
import numpy as np


### Working Directory

This block sets the main project paths so file loading works the same way each time. The layout follows notebook 01.


In [2]:
# this block sets the main project paths so file loading works the same way each time.
def find_project_root(start_path=None):
    start_path = Path.cwd().resolve() if start_path is None else Path(start_path).resolve()
    for candidate in [start_path, *start_path.parents]:
        if (candidate / "scripts" / "final_final_scripts").exists() and (candidate / "data").exists():
            return candidate
    raise FileNotFoundError("Could not find project root from the current working directory.")

here = Path.cwd().resolve()
project_root = find_project_root(here)
data_dir = project_root / "data" / "TokaGabor"

print("here:", here)
print("project_root:", project_root)
print("data_dir:", data_dir)


here: /Users/andrasikristof/Documents/Egyetem/2026:27 - 2. félév/Election_predictions/scripts/final_final_scripts
project_root: /Users/andrasikristof/Documents/Egyetem/2026:27 - 2. félév/Election_predictions
data_dir: /Users/andrasikristof/Documents/Egyetem/2026:27 - 2. félév/Election_predictions/data/TokaGabor


Interpretation. The printed paths should point to this election project. If they do not, the later loading steps can fail.


## Notebook 14

This notebook is prepared for the day after the election. It keeps the notebook opening consistent with the other files, stores the final notebook 13 forecast, and makes it easy to compare that forecast with the real result once the official numbers are available.


### Imports and Constants

This block loads the extra tools used only in this notebook. It also sets the block order and keeps the notebook 13 main path name visible.


In [3]:
# this block loads the extra tools used in notebook 14.
import json
from IPython.display import display

# this list keeps the political blocks in one fixed order everywhere.
block_cols = ["gov", "opp", "opp_radical", "other"]

# this is the main written forecast from notebook 13.
main_path_name = "simple_weighted_average"
sensitivity_path_name = "house_effects_rw"
notebook_13_path = here / "13_final_2026_forecast.ipynb"
notebook_13_export_path = here / "13_final_2026_forecast_export.json"


### Store The Notebook 13 Forecast

This block loads the forecast that notebook 13 reported. It first tries to read the machine-readable export created by notebook 13, and if that file is not available yet it tries to read the saved outputs from `13_final_2026_forecast.ipynb`.


In [4]:
# this block loads the forecast values from notebook 13.
required_path_names = [main_path_name, sensitivity_path_name]
required_vote_keys = block_cols.copy()
required_seat_keys = [block for block in block_cols if block != "other"]
required_probability_keys = [
    "gov finishes first",
    "opp finishes first",
    "gov majority",
    "gov two thirds",
    "opp majority",
    "opp two thirds",
    "parliament without gov majority",
    "parliament without opp majority",
]


def cell_output_text(cell):
    # this function joins every text output from one code cell.
    pieces = []
    for output in cell.get("outputs", []):
        if "text" in output:
            pieces.append("".join(output["text"]))
        elif "data" in output and "text/plain" in output["data"]:
            pieces.append("".join(output["data"]["text/plain"]))
    return "\n".join(pieces)


def parse_vote_share(text, label):
    # this function reads one printed vote-share sentence from notebook 13.
    pattern = (
        rf"{label}: ([0-9.]+)% gov, ([0-9.]+)% opp, "
        rf"([0-9.]+)% opp_radical, and ([0-9.]+)% other\."
    )
    match = re.search(pattern, text)
    if match is None:
        return None
    return {
        "gov": float(match.group(1)),
        "opp": float(match.group(2)),
        "opp_radical": float(match.group(3)),
        "other": float(match.group(4)),
    }


def parse_point_seats(text, label):
    # this function reads one printed point-seat sentence from notebook 13.
    pattern = rf"{label}: gov ([0-9]+), opp ([0-9]+), opp_radical ([0-9]+)(?:, other ([0-9]+))?\."
    match = re.search(pattern, text)
    if match is None:
        return None
    return {
        "gov": int(match.group(1)),
        "opp": int(match.group(2)),
        "opp_radical": int(match.group(3)),
        "other": int(match.group(4)) if match.group(4) is not None else 0,
    }


def parse_expected_seats(text, label):
    # this function reads one printed expected-seat sentence from notebook 13.
    pattern = rf"{label}: gov ([0-9.]+), opp ([0-9.]+), opp_radical ([0-9.]+)(?:, other ([0-9.]+))?\."
    match = re.search(pattern, text)
    if match is None:
        return None
    return {
        "gov": float(match.group(1)),
        "opp": float(match.group(2)),
        "opp_radical": float(match.group(3)),
        "other": float(match.group(4)) if match.group(4) is not None else 0.0,
    }


def parse_probabilities(text, label):
    # this function reads one printed probability sentence from notebook 13.
    pattern = (
        rf"{label}: gov first ([0-9.]+)%, opp first ([0-9.]+)%, "
        rf"gov majority ([0-9.]+)%, gov two thirds ([0-9.]+)%, "
        rf"parliament without gov majority ([0-9.]+)%, "
        rf"opp majority ([0-9.]+)%, opp two thirds ([0-9.]+)%, "
        rf"parliament without opp majority ([0-9.]+)%\."
    )
    match = re.search(pattern, text)
    if match is None:
        return None
    return {
        "gov finishes first": float(match.group(1)),
        "opp finishes first": float(match.group(2)),
        "gov majority": float(match.group(3)),
        "gov two thirds": float(match.group(4)),
        "parliament without gov majority": float(match.group(5)),
        "opp majority": float(match.group(6)),
        "opp two thirds": float(match.group(7)),
        "parliament without opp majority": float(match.group(8)),
    }


def validate_prediction_bundle(bundle):
    # this function checks that the loaded forecast has the fields notebook 14 needs.
    for path_name in required_path_names:
        if path_name not in bundle:
            raise ValueError(f"Missing forecast path: {path_name}")

        path_bundle = bundle[path_name]

        for key in ["vote_share_pct", "point_seats", "expected_seats", "probabilities_pct"]:
            if key not in path_bundle:
                raise ValueError(f"Missing key '{key}' inside path '{path_name}'")

        path_bundle["point_seats"] = {
            **{block: path_bundle["point_seats"][block] for block in required_seat_keys},
            "other": path_bundle["point_seats"].get("other", 0),
        }
        path_bundle["expected_seats"] = {
            **{block: path_bundle["expected_seats"][block] for block in required_seat_keys},
            "other": path_bundle["expected_seats"].get("other", 0.0),
        }

        for block in required_vote_keys:
            _ = float(path_bundle["vote_share_pct"][block])
            _ = float(path_bundle["expected_seats"][block])
            _ = float(path_bundle["point_seats"][block])

        for event_name in required_probability_keys:
            _ = float(path_bundle["probabilities_pct"][event_name])

    return bundle


def load_prediction_export(export_path):
    # this function loads the machine-readable export that notebook 13 saves.
    if not export_path.exists():
        return None, "notebook 13 export file not found yet"

    with export_path.open(encoding="utf-8") as f:
        export_data = json.load(f)

    path_bundle = export_data.get("paths")
    if path_bundle is None:
        raise ValueError("The notebook 13 export file exists, but it does not contain a 'paths' section.")

    return validate_prediction_bundle(path_bundle), "loaded from notebook 13 export file"


def load_notebook_13_predictions_from_outputs(path):
    # this function recovers the saved printed summary from notebook 13 if needed.
    if not path.exists():
        return None, "notebook 13 file was not found"

    with path.open(encoding="utf-8") as f:
        notebook = json.load(f)

    full_text = "\n".join(
        cell_output_text(cell)
        for cell in notebook.get("cells", [])
        if cell.get("cell_type") == "code"
    )

    def parse_first_match(parse_function, labels):
        # this function tries several old and new notebook 13 labels.
        for label in labels:
            parsed = parse_function(full_text, label)
            if parsed is not None:
                return parsed
        return None

    simple_vote = parse_first_match(parse_vote_share, ["Simple path combined forecast", "Simple 70/30 combined forecast"])
    house_vote = parse_first_match(parse_vote_share, ["House-effects RW combined forecast", "House-effects RW 70/30 combined forecast"])
    simple_point = parse_first_match(parse_point_seats, ["Simple path point seats", "Simple 70/30 path point seats"])
    house_point = parse_first_match(parse_point_seats, ["House-effects RW point seats", "House-effects RW 70/30 path point seats"])
    simple_expected = parse_first_match(parse_expected_seats, ["Simple path expected seats from simulation", "Simple 70/30 path expected seats from simulation"])
    house_expected = parse_first_match(parse_expected_seats, ["House-effects RW expected seats from simulation", "House-effects RW 70/30 path expected seats from simulation"])
    simple_prob = parse_first_match(parse_probabilities, ["Simple path simulation", "Simple 70/30 path simulation"])
    house_prob = parse_first_match(parse_probabilities, ["House-effects RW simulation", "House-effects RW 70/30 path simulation"])

    parsed_ok = all(
        item is not None
        for item in [
            simple_vote,
            house_vote,
            simple_point,
            house_point,
            simple_expected,
            house_expected,
            simple_prob,
            house_prob,
        ]
    )

    if not parsed_ok:
        return None, "notebook 13 saved outputs were incomplete"

    parsed_predictions = {
        "simple_weighted_average": {
            "vote_share_pct": simple_vote,
            "point_seats": simple_point,
            "expected_seats": simple_expected,
            "probabilities_pct": simple_prob,
        },
        "house_effects_rw": {
            "vote_share_pct": house_vote,
            "point_seats": house_point,
            "expected_seats": house_expected,
            "probabilities_pct": house_prob,
        },
    }
    return validate_prediction_bundle(parsed_predictions), "loaded from notebook 13 saved outputs"


prediction_bundle = None
prediction_source = None

try:
    prediction_bundle, prediction_source = load_prediction_export(notebook_13_export_path)
except Exception as export_error:
    print(f"Notebook 13 export file could not be used: {export_error}")

if prediction_bundle is None:
    output_bundle, output_source = load_notebook_13_predictions_from_outputs(notebook_13_path)
    prediction_bundle = output_bundle
    prediction_source = output_source

if prediction_bundle is None:
    raise FileNotFoundError(
        "Notebook 14 could not load notebook 13 predictions. "
        "Run notebook 13, save it, and make sure its export cell creates 13_final_2026_forecast_export.json."
    )

prediction_vote_share = pd.DataFrame(
    [
        {"path": path_name, **prediction_bundle[path_name]["vote_share_pct"]}
        for path_name in [main_path_name, sensitivity_path_name]
    ]
)
prediction_point_seats = pd.DataFrame(
    [
        {"path": path_name, **prediction_bundle[path_name]["point_seats"]}
        for path_name in [main_path_name, sensitivity_path_name]
    ]
)
prediction_expected_seats = pd.DataFrame(
    [
        {"path": path_name, **prediction_bundle[path_name]["expected_seats"]}
        for path_name in [main_path_name, sensitivity_path_name]
    ]
)
prediction_source_table = pd.DataFrame(
    {
        "item": ["main path", "sensitivity path", "prediction source", "prediction notebook"],
        "value": [main_path_name, sensitivity_path_name, prediction_source, str(notebook_13_path)],
    }
)

display(prediction_source_table)
display(prediction_vote_share.round(2))
display(prediction_point_seats)
display(prediction_expected_seats.round(2))


,item,value
0,main path,simple_weighted_average
1,sensitivity path,house_effects_rw
2,prediction source,loaded from notebook 13 export file
3,prediction notebook,/Users/andrasikristof/Documents/Egyetem/2026:2...


,path,gov,opp,opp_radical,other
0,simple_weighted_average,43.18,44.11,7.04,5.67
1,house_effects_rw,42.98,44.60,7.73,4.69


,path,gov,opp,opp_radical,other
0,simple_weighted_average,102,89,8,0
1,house_effects_rw,97,93,9,0


,path,gov,opp,opp_radical,other
0,simple_weighted_average,94.53,90.64,13.83,0.0
1,house_effects_rw,90.65,92.19,16.15,0.0


**Interpretation:** The first table shows where the forecast numbers came from. The next tables show the notebook 13 predictions that notebook 14 will compare against, with `simple_weighted_average` treated as the main forecast and `house_effects_rw` kept as a sensitivity check.


### Enter The Actual Election Results

This block is the only place you need to edit after the election. Replace each `np.nan` with the official block-level result, keeping vote shares in percent form like `43.6` and seat counts as whole numbers like `104`.


In [5]:
# this block creates one simple input area for the real election result.
# when the official result is available, replace np.nan with the real values.
actual_vote_share_input = {
    "gov": 38.43,
    "opp": 53.07,
    "opp_radical": 5.83,
    "other": 2.67,
}

actual_total_seats_input = {
    "gov": 55,
    "opp": 138,
    "opp_radical": 6,
    "other": 0,
}

actual_vote_share = pd.DataFrame(
    {"block": block_cols, "actual_vote_share_pct": [actual_vote_share_input[block] for block in block_cols]}
)
actual_total_seats = pd.DataFrame(
    {"block": block_cols, "actual_total_seats": [actual_total_seats_input[block] for block in block_cols]}
)

actual_input_status = pd.DataFrame(
    {
        "input_table": ["actual vote share", "actual total seats"],
        "missing_values": [
            int(actual_vote_share["actual_vote_share_pct"].isna().sum()),
            int(actual_total_seats["actual_total_seats"].isna().sum()),
        ],
    }
)

display(actual_vote_share)
display(actual_total_seats)
display(actual_input_status)


,block,actual_vote_share_pct
0,gov,38.43
1,opp,53.07
2,opp_radical,5.83
3,other,2.67


,block,actual_total_seats
0,gov,55
1,opp,138
2,opp_radical,6
3,other,0


,input_table,missing_values
0,actual vote share,0
1,actual total seats,0


**Interpretation:** Before election day, it is normal that the tables are empty and the missing-value counts are four. After you fill this block, both missing-value counts should become zero.


### Helper Functions

This block defines the small functions used in the comparison cells. It keeps the later blocks short and easier to read.


In [6]:
# this block defines helper functions for the comparison tables.
def make_error_table(prediction_map, actual_map, value_label):
    # this function compares one prediction map with one actual map.
    predicted = pd.Series(prediction_map, name="predicted")
    actual = pd.Series(actual_map, name="actual")
    table = pd.concat([predicted, actual], axis=1).reset_index()
    table.columns = ["block", "predicted", "actual"]
    table["signed_error"] = table["predicted"] - table["actual"]
    table["abs_error"] = table["signed_error"].abs()
    table["metric"] = value_label
    return table


def make_summary_row(path_name, comparison_table):
    # this function builds one short summary row from an error table.
    mae = float(comparison_table["abs_error"].mean())
    rmse = float(np.sqrt(np.mean(comparison_table["signed_error"] ** 2)))
    max_block = comparison_table.sort_values("abs_error", ascending=False).iloc[0]["block"]
    max_abs_error = float(comparison_table["abs_error"].max())
    return {
        "path": path_name,
        "mae": mae,
        "rmse": rmse,
        "largest_miss_block": max_block,
        "largest_abs_error": max_abs_error,
    }


def input_ready(frame, value_col):
    # this function checks if one input table is fully filled.
    return bool(frame[value_col].notna().all())


def build_actual_event_map(actual_vote_map, actual_seat_map):
    # this function turns the real result into true or false event outcomes.
    return {
        "gov finishes first": actual_vote_map["gov"] > max(actual_vote_map["opp"], actual_vote_map["opp_radical"], actual_vote_map["other"]),
        "opp finishes first": actual_vote_map["opp"] > max(actual_vote_map["gov"], actual_vote_map["opp_radical"], actual_vote_map["other"]),
        "gov majority": actual_seat_map["gov"] >= 100,
        "gov two thirds": actual_seat_map["gov"] >= 133,
        "opp majority": actual_seat_map["opp"] >= 100,
        "opp two thirds": actual_seat_map["opp"] >= 133,
        "parliament without gov majority": actual_seat_map["gov"] < 100,
        "parliament without opp majority": actual_seat_map["opp"] < 100,
    }


def make_probability_table(probability_map, actual_event_map):
    # this function checks how well the forecast probabilities matched the real event.
    table = pd.DataFrame(
        {
            "event": list(probability_map.keys()),
            "forecast_probability_pct": list(probability_map.values()),
        }
    )
    table["actual_happened"] = table["event"].map(actual_event_map).astype(int)
    table["brier_component"] = ((table["forecast_probability_pct"] / 100.0) - table["actual_happened"]) ** 2
    return table


### Vote Share Accuracy

This block compares the notebook 13 predicted national block vote shares with the real national block vote shares. It reports signed error and absolute error in percentage points.


In [7]:
# this block compares forecast vote shares with the real vote shares.
if input_ready(actual_vote_share, "actual_vote_share_pct"):
    actual_vote_map = actual_vote_share.set_index("block")["actual_vote_share_pct"].to_dict()

    vote_share_tables = []
    vote_share_summary_rows = []

    for path_name in [main_path_name, sensitivity_path_name]:
        path_table = make_error_table(
            prediction_bundle[path_name]["vote_share_pct"],
            actual_vote_map,
            value_label="vote_share_pct",
        )
        path_table["path"] = path_name
        vote_share_tables.append(path_table)
        vote_share_summary_rows.append(make_summary_row(path_name, path_table))

    vote_share_comparison = pd.concat(vote_share_tables, ignore_index=True)
    vote_share_summary = pd.DataFrame(vote_share_summary_rows)

    actual_vote_winner = max(actual_vote_map, key=actual_vote_map.get)
    vote_share_summary["predicted_winner"] = vote_share_summary["path"].map(
        lambda path_name: max(prediction_bundle[path_name]["vote_share_pct"], key=prediction_bundle[path_name]["vote_share_pct"].get)
    )
    vote_share_summary["actual_winner"] = actual_vote_winner
    vote_share_summary["winner_correct"] = vote_share_summary["predicted_winner"] == vote_share_summary["actual_winner"]

    display(vote_share_comparison.round(2))
    display(vote_share_summary.round(2))
else:
    print("Fill the actual vote share block above, then run this cell again.")


,block,predicted,actual,signed_error,abs_error,metric,path
0,gov,43.18,38.43,4.75,4.75,vote_share_pct,simple_weighted_average
1,opp,44.11,53.07,-8.96,8.96,vote_share_pct,simple_weighted_average
2,opp_radical,7.04,5.83,1.21,1.21,vote_share_pct,simple_weighted_average
3,other,5.67,2.67,3.00,3.00,vote_share_pct,simple_weighted_average
4,gov,42.98,38.43,4.55,4.55,vote_share_pct,house_effects_rw
5,opp,44.60,53.07,-8.47,8.47,vote_share_pct,house_effects_rw
6,opp_radical,7.73,5.83,1.90,1.90,vote_share_pct,house_effects_rw
7,other,4.69,2.67,2.02,2.02,vote_share_pct,house_effects_rw


,path,mae,rmse,largest_miss_block,largest_abs_error,predicted_winner,actual_winner,winner_correct
0,simple_weighted_average,4.48,5.32,opp,8.96,opp,opp,True
1,house_effects_rw,4.23,5.00,opp,8.47,opp,opp,True


**Interpretation:** Lower `abs_error` means the forecast was closer to the real vote share. In the summary table, lower `mae` and `rmse` are better, and `winner_correct = True` means that path called the winning block correctly.


### Seat Accuracy

This block checks two seat targets against the real seat result: the point seat forecast and the expected seat forecast from the simulation. This helps you see whether the single central scenario or the simulation average was closer to reality.


In [8]:
# this block compares forecast seats with the real seat totals.
if input_ready(actual_total_seats, "actual_total_seats"):
    actual_seat_map = actual_total_seats.set_index("block")["actual_total_seats"].to_dict()

    seat_tables = []
    seat_summary_rows = []

    for path_name in [main_path_name, sensitivity_path_name]:
        for seat_type in ["point_seats", "expected_seats"]:
            path_table = make_error_table(
                prediction_bundle[path_name][seat_type],
                actual_seat_map,
                value_label=seat_type,
            )
            path_table["path"] = path_name
            seat_tables.append(path_table)

            summary_row = make_summary_row(path_name, path_table)
            summary_row["seat_target"] = seat_type
            seat_summary_rows.append(summary_row)

    seat_comparison = pd.concat(seat_tables, ignore_index=True)
    seat_summary = pd.DataFrame(seat_summary_rows)

    display(seat_comparison.round(2))
    display(seat_summary.round(2))
else:
    print("Fill the actual total seats block above, then run this cell again.")


,block,predicted,actual,signed_error,abs_error,metric,path
0,gov,102.00,55,47.00,47.00,point_seats,simple_weighted_average
1,opp,89.00,138,-49.00,49.00,point_seats,simple_weighted_average
2,opp_radical,8.00,6,2.00,2.00,point_seats,simple_weighted_average
3,other,0.00,0,0.00,0.00,point_seats,simple_weighted_average
4,gov,94.53,55,39.53,39.53,expected_seats,simple_weighted_average
5,opp,90.64,138,-47.36,47.36,expected_seats,simple_weighted_average
6,opp_radical,13.83,6,7.83,7.83,expected_seats,simple_weighted_average
7,other,0.00,0,0.00,0.00,expected_seats,simple_weighted_average
8,gov,97.00,55,42.00,42.00,point_seats,house_effects_rw
9,opp,93.00,138,-45.00,45.00,point_seats,house_effects_rw


,path,mae,rmse,largest_miss_block,largest_abs_error,seat_target
0,simple_weighted_average,24.50,33.96,opp,49.00,point_seats
1,simple_weighted_average,23.68,31.09,opp,47.36,expected_seats
2,house_effects_rw,22.50,30.81,opp,45.00,point_seats
3,house_effects_rw,22.90,29.46,opp,45.81,expected_seats


**Interpretation:** Lower seat errors mean the forecast translated votes into parliament more accurately. Compare `point_seats` and `expected_seats` in the summary table to see which target was closer overall.


### Probability Check

This block compares the notebook 13 event probabilities with what actually happened. It builds simple true-or-false events from the real vote-share winner and the real seat totals, then calculates a Brier-style score where lower is better.


In [9]:
# this block checks the forecast probabilities against the real result.
if input_ready(actual_vote_share, "actual_vote_share_pct") and input_ready(actual_total_seats, "actual_total_seats"):
    actual_vote_map = actual_vote_share.set_index("block")["actual_vote_share_pct"].to_dict()
    actual_seat_map = actual_total_seats.set_index("block")["actual_total_seats"].to_dict()
    actual_event_map = build_actual_event_map(actual_vote_map, actual_seat_map)

    probability_tables = []
    probability_summary_rows = []

    for path_name in [main_path_name, sensitivity_path_name]:
        path_table = make_probability_table(
            prediction_bundle[path_name]["probabilities_pct"],
            actual_event_map,
        )
        path_table["path"] = path_name
        probability_tables.append(path_table)
        probability_summary_rows.append(
            {
                "path": path_name,
                "mean_brier_score": float(path_table["brier_component"].mean()),
            }
        )

    probability_comparison = pd.concat(probability_tables, ignore_index=True)
    probability_summary = pd.DataFrame(probability_summary_rows)

    display(probability_comparison.round(4))
    display(probability_summary.round(4))
else:
    print("Fill both actual input blocks above, then run this cell again.")


,event,forecast_probability_pct,actual_happened,brier_component,path
0,gov finishes first,46.6000,0,0.2172,simple_weighted_average
1,opp finishes first,53.4000,1,0.2172,simple_weighted_average
2,gov majority,39.2667,0,0.1542,simple_weighted_average
3,gov two thirds,1.0667,0,0.0001,simple_weighted_average
4,opp majority,29.7333,1,0.4937,simple_weighted_average
5,opp two thirds,1.4000,1,0.9722,simple_weighted_average
6,parliament without gov majority,60.7333,1,0.1542,simple_weighted_average
7,parliament without opp majority,70.2667,0,0.4937,simple_weighted_average
8,gov finishes first,46.6000,0,0.2172,house_effects_rw
9,opp finishes first,53.2667,1,0.2184,house_effects_rw


,path,mean_brier_score
0,simple_weighted_average,0.3378
1,house_effects_rw,0.2666


**Interpretation:** A lower `brier_component` means the forecast probability matched the real event better. The `mean_brier_score` gives one overall number for each path, where lower is better.


### Final Easy English Summary

This block prints one short summary sentence after the real results are entered. It is meant to give you a fast answer to the question: how far off was notebook 13?


In [10]:
# this block prints a short easy English summary.
if input_ready(actual_vote_share, "actual_vote_share_pct") and input_ready(actual_total_seats, "actual_total_seats"):
    actual_vote_map = actual_vote_share.set_index("block")["actual_vote_share_pct"].to_dict()
    actual_seat_map = actual_total_seats.set_index("block")["actual_total_seats"].to_dict()

    main_vote_table = make_error_table(
        prediction_bundle[main_path_name]["vote_share_pct"],
        actual_vote_map,
        value_label="vote_share_pct",
    )
    main_seat_table = make_error_table(
        prediction_bundle[main_path_name]["point_seats"],
        actual_seat_map,
        value_label="point_seats",
    )

    main_vote_mae = float(main_vote_table["abs_error"].mean())
    main_seat_mae = float(main_seat_table["abs_error"].mean())
    worst_vote_row = main_vote_table.sort_values("abs_error", ascending=False).iloc[0]
    predicted_winner = max(prediction_bundle[main_path_name]["vote_share_pct"], key=prediction_bundle[main_path_name]["vote_share_pct"].get)
    actual_winner = max(actual_vote_map, key=actual_vote_map.get)

    print(
        f"Notebook 13 main path ({main_path_name}) missed the national block vote shares by "
        f"{main_vote_mae:.2f} percentage points on average."
    )
    print(
        f"It missed the final seat totals by {main_seat_mae:.2f} seats on average."
    )
    print(
        f"The largest vote-share miss was {worst_vote_row['block']} with an absolute error of "
        f"{worst_vote_row['abs_error']:.2f} percentage points."
    )
    print(
        f"The predicted winning block was {predicted_winner}, while the actual winning block was {actual_winner}."
    )
else:
    print("The notebook is ready. Fill the actual result input block after election day, then rerun the comparison cells.")


Notebook 13 main path (simple_weighted_average) missed the national block vote shares by 4.48 percentage points on average.
It missed the final seat totals by 24.50 seats on average.
The largest vote-share miss was opp with an absolute error of 8.96 percentage points.
The predicted winning block was opp, while the actual winning block was opp.


**Interpretation:** If the actual data is not entered yet, this cell simply confirms that the notebook is ready. After the election, it gives a short plain-language summary of the main forecast error.


## Final Note

This notebook is designed to be easy to use on election night or the next morning:

- keep notebook 13 as the forecast source
- enter the official block-level vote shares and seat totals in one place
- rerun the comparison cells
- read the error tables and the final easy English summary
